In [7]:
import sys
import os
import pathlib as pl
sys.path.append('..')
home = pl.Path(os.getcwd())

In [8]:
import glob

In [9]:
#from src.core import *

import pandas as pd
from pydsstools.heclib.dss import HecDss
from pydsstools.core import TimeSeriesContainer
import datetime


## Get HUC basins

In [10]:
inputs_dir = home/'input'
outputs_dir = home/'output'

In [17]:
#user defined
project = 'steer'
subfolder = 'site_6'

assert os.path.exists(home/'input'/project/subfolder), 'Must put hydroCAD files in a subfolder'


In [18]:
event_dict = {'A': '1yr', 'B': '2yr','C': '5yr','D': '10yr','E': '25yr','F': '50yr','G': '100yr'}
start_date = '01JAN2026 00:00:00'
time_incr = '3MIN'

In [19]:
sub_catch = glob.glob(str(home/'input'/project/subfolder)+'/*.csv')

In [20]:
sub_catch

['c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 10.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 12.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 13.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 14.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 16.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 17.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 18.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 3.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 4.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 6.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 8.csv',
 'c:\\git\\hydromet\\notebooks\\input\\steer\\site_6\\Site 6~Subcat 9.csv']

In [21]:
## TODO set up way to read the time increment from the data rather than manual set

In [22]:
from pydsstools.heclib.dss import HecDss
from pydsstools.core import TimeSeriesContainer

In [23]:

# 1. Read CSV data
for sub_catch_site in sub_catch:
    df = pd.read_csv(sub_catch_site)
    df = df.rename(columns=lambda x: x.replace('\n', ''))
    #get flow intervals
    cols = list(df.columns[1:])
    # 2. Open/Create DSS file
    dss_file = home/'input'/project/f"{subfolder}.dss"
    node = sub_catch_site[sub_catch_site.find('~')+1:sub_catch_site.find('.csv')].replace(' ','_')
    with HecDss.Open(str(dss_file)) as fid:

        # 3. Define Pathname Parts
        # /A/B/C/D/E/F/
        for col in cols:
            pathname = f"/{project}/{node}/FLOW//{time_incr}/{col}/"
            count = len(df[col].values)
            interval = 1
            # 4. Prepare TimeSeriesContainer
            tsc = TimeSeriesContainer()
            tsc.pathname = pathname
            tsc.startDateTime = start_date
            tsc.numberValues =count 
            tsc.units = "cfs"
            tsc.type = "INST"
            tsc.values = df[col].values
            fid.deletePathname(tsc.pathname)
            # 5. Write to DSS
            fid.put_ts(tsc)
            print(f"Data written to {dss_file} with path {pathname}")

Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/A-1yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/B-2yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/C-5yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/D-10yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/E-25yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/F-50yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_10/FLOW//3MIN/G-100yrRunoff(cfs)/
Data written to c:\git\hydromet\notebooks\input\steer\site_6.dss with path /steer/Subcat_12/FLOW//3MIN/A-1yrRunoff(cfs)/
Data written to c:\git\hydr

In [24]:
## END